<!-- nav -->
[← 1 · Programs as data](01_Programs_As_Data.ipynb) · [Home](../../README.md) · [3 · Which standard? →](03_Which_Standard.ipynb)

# 2 · Changing code safely

**The problem.** A team replaces `print` with a logger across a code base. A regular expression finds `print(` in
strings, comments and names like `blueprint(`, and misses `print (`. Editing code as text is guessing at its structure.

On a tree, a call to `print` is a `Call` whose function is the `Name` `print`: there's nothing to guess. This case
study finds, changes, replaces and copies syntax nodes.

In [ ]:
from mbse.Programs.Framework.Syntax import Parents, Transformer, copy, walk
from mbse.Programs.Python import Python314, Syntax as P

Py = P.LANGUAGE.Builders
source = '''def report(total):
    print("total:", total)
    blueprint("print(total)")
    return total


print(report(42))
'''
module = Python314.parse(source)

## Step 1: finding

`walk` visits every syntax node of a tree, parents before children. A call to `print` is easy to say exactly:

In [ ]:
def calls_to(name, tree):
    return [node for node in walk(tree)
            if isinstance(node, P.Call) and isinstance(node.func, P.Name) and node.func.id.spelling == name]


print(len(calls_to("print", module)), "calls to print")
print([Python314.print(call.args[0]).strip() for call in calls_to("print", module)])

Two, and not `blueprint`, nor the `print(total)` inside the string.

## Step 2: changing in place

Syntax nodes are plain objects, so the simplest change is an assignment. Each `print` becomes `log.info`, by replacing
the call's function:

In [ ]:
for call in calls_to("print", module):
    call.func = Py.Attribute().value(lambda b: b.Name().id("log")).attr("info").create()
print(Python314.print(module))

## Step 3: a transformer, kind by kind

For changes all over a tree, a `Transformer` has a `visit_<Kind>` method per kind it changes, and returns what replaces
each syntax node. Here every `return x` gains a log line before it. A statement can be replaced by several, so the
method returns a list:

In [ ]:
class LogReturns(Transformer):
    def visit_Return(self, node):
        logged = Py.Expr().value(lambda b: b.Call().func(lambda b: b.Attribute().value(lambda b: b.Name().id("log"))
                                                         .attr("debug")).add_args(copy(node.value))).create()
        return [logged, node]


LogReturns().visit(module)
print(Python314.print(module))

`copy(node.value)` duplicates the returned expression, because the log line and the return can't share one syntax
node: step 5 says why.

## Step 4: where a syntax node is

`Parents` records each syntax node's parent, the property holding it and its index, and its **path** from the root,
which is how errors name places in a tree. It also replaces and removes:

In [ ]:
parents = Parents(module)
blueprint = next(node for node in walk(module) if isinstance(node, P.Name) and node.id.spelling == "blueprint")
statement = parents.parent(parents.parent(blueprint))
print(parents.path(blueprint), "in", parents.path(statement))
parents.remove(statement)
print(Python314.print(module))

## Step 5: one place per syntax node

A tree is a tree: each syntax node has one parent. Using a syntax node twice makes two places that would change
together, and `validate` refuses it. `copy` makes the second one:

In [ ]:
name = Py.Name().id("n").create()
shared = Py.BinOp().left(name).op("*").right(name).create()
print(P.LANGUAGE.validate(shared))
squared = Py.BinOp().left(name).op("*").right(copy(name)).create()
print(P.LANGUAGE.validate(squared), Python314.print(squared))

## Why it works this way

- **Structure, not spelling.** A call is a `Call`, whatever its spacing; a string is a `Constant`, whatever it contains.
  Changes made on the tree can't touch what they don't mean to.
- **Several tools, one tree.** `walk` for finding, assignment for small changes, `Transformer` for sweeping ones,
  `Parents` for where things are: each the right size for its job, all on the same plain objects.
- **One parent each.** Two places that silently share a syntax node would change together. Copying is explicit.

Renaming is harder than it looks: the next case studies cover which standard a change needs, and then what names refer
to.

<!-- nav -->
[← 1 · Programs as data](01_Programs_As_Data.ipynb) · [Home](../../README.md) · [3 · Which standard? →](03_Which_Standard.ipynb)